<a href="https://colab.research.google.com/github/Riju0045/Face-Emotion-Recognition/blob/main/Face_Emotion_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/archive (9).zip" -d "/content/"

In [3]:
tren = "/content/archive (3)/Train"
tst = "/content/archive (3)/Test"

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense, GlobalAveragePooling2D, Input, Lambda
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    tren,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,

)

test_ds = tf.keras.utils.image_dataset_from_directory(
    tst,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,

)


AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model2 = Sequential([
    Input(shape=(224, 224, 3)),
    data_augmentation,
    Lambda(preprocess_input),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(8, activation="softmax")
])

model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

history1 = model2.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[early_stop]
)


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2
)

history2 = model2.fit(
    train_ds,
    validation_data=test_ds,
    epochs=45,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report
yt= np.concatenate([y for x, y in test_ds], axis=0)
yp= np.argmax(model2.predict(test_ds),axis=1)

In [ ]:
print(classification_report(yt,yp))


In [ ]:
import os
cn = sorted(os.listdir(tst))
print(cn)
print(len(cn))
print(cn)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(yt, yp)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=cn, yticklabels=cn, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y_train = np.concatenate([
    labels.numpy() for _, labels in train_ds
])
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

model2.fit(train_ds, validation_data=test_ds, epochs=35,
           callbacks=[early_stop, reduce_lr],
           class_weight=class_weight_dict)

In [ ]:
yt1= np.concatenate([y for x, y in test_ds], axis=0)
yp1= np.argmax(model2.predict(test_ds),axis=1)

In [ ]:
print(classification_report(yt1,yp1))

In [ ]:
cm = confusion_matrix(yt1, yp1)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=cn, yticklabels=cn, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()